In [5]:
!pip install -q -U google-genai

In [4]:
!pip install -q gradio

In [3]:
import pandas as pd

# 讀取您上傳的 CDP 資料表 (設定編碼為 big5 以支援中文)
df = pd.read_csv('CDP資料.csv', encoding='big5')

# 假設今天登入的車主編號是 133
user_id = 133

# 從資料庫中把這位車主的資料撈出來
user_data = df[df['編號'] == user_id].iloc[0]

car_model = user_data['現保有車款_T']      # 抓出: bZ4X
location = user_data['居住縣市']            # 抓出: 新北市
maintenance_count = user_data['近一年回廠有費定保次數\n(定期保養)']

# 接著就把這些變數放進 Prompt 裡給 AI ...

FileNotFoundError: [Errno 2] No such file or directory: 'CDP資料.csv'

In [ ]:
import gradio as gr
from google import genai
from google.colab import userdata

# 1. 系統初始化
try:
    api_key = userdata.get('api')
    client = genai.Client(api_key=api_key)
except Exception as e:
    print(f"API Key 讀取失敗: {e}")

# 2. 確認建立檔案
def confirm_profile(name, car, mileage, pref):
    if not name or not car:
        return "⚠️ 請至少填寫您的稱呼與車款！", gr.update(interactive=False)
    success_msg = f"✅ 檔案建立成功！您好，{name}。\n系統已紀錄您的愛車為 {car}，請在右側開始對話。"
    return success_msg, gr.update(interactive=True, placeholder="請輸入您的問題並按 Enter 送出...")

# 3. 對話邏輯（messages 格式）
def _text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(c.get("text", "") if isinstance(c, dict) else str(c) for c in content)
    if isinstance(content, dict):
        return content.get("text", "")
    return str(content)

def chat_with_ai(user_message, history, name, car, mileage, pref):
    history = history or []
    if not user_message or not user_message.strip():
        return "", history

    prompt = f"""
你現在是 TOYOTA 專屬的智能隨行顧問。
請以親切、專業的語氣解答車主的問題。

以下是車主剛剛建立的檔案，請務必根據這些條件給予最適合他的建議：
- 車主姓名：{name}
- 車款：{car}
- 目前里程數：{mileage}
- 用車習慣/喜好：{pref}

--- 歷史對話紀錄 ---
"""
    for m in history:
        who = "車主" if m.get("role") == "user" else "顧問"
        prompt += f"{who}：{_text(m.get('content'))}\n"
    prompt += f"車主：{user_message}\n顧問："

    try:
        response = client.models.generate_content(
            model="gemini-3.8-flash",
            contents=prompt
        )
        reply = response.text
    except Exception as e:
        reply = f"伺服器錯誤，請稍後再試。({e})"

    history = history + [
        {"role": "user", "content": user_message},
        {"role": "assistant", "content": reply},
    ]
    return "", history

# 4. 介面
with gr.Blocks(theme=gr.themes.Base()) as demo:
    gr.Markdown("# 🚗 TOYOTA SmartLife 專屬智能顧問")
    gr.Markdown("請先在左側建立您的專屬檔案，接著即可在右側開始體驗客製化的 AI 隨行服務。")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📋 步驟 1：建立車主檔案")
            name_input = gr.Textbox(label="您的稱呼", placeholder="例如：魏先生")
            car_input = gr.Textbox(label="您的愛車車款", placeholder="例如：bZ4X")
            mileage_input = gr.Textbox(label="目前大約里程數", placeholder="例如：23000")
            pref_input = gr.Textbox(label="您的用車習慣與喜好", placeholder="例如：每天通勤")
            confirm_btn = gr.Button("✅ 確認並建立檔案", variant="primary")
            status_text = gr.Textbox(label="系統狀態", interactive=False)

        with gr.Column(scale=2):
            gr.Markdown("### 💬 步驟 2：智能對話區")
            try:
                chatbot = gr.Chatbot(label="顧問回覆", height=450, type="messages")
            except TypeError:  # 新版 Gradio 已移除 type 參數，預設即 messages 格式
                chatbot = gr.Chatbot(label="顧問回覆", height=450)

            with gr.Row():
                msg_input = gr.Textbox(
                    label="請輸入您的問題",
                    placeholder="請先在左側點擊「確認並建立檔案」...",
                    scale=4,
                    interactive=False
                )
                submit_btn = gr.Button("📤 送出", scale=1)

    confirm_btn.click(
        confirm_profile,
        inputs=[name_input, car_input, mileage_input, pref_input],
        outputs=[status_text, msg_input]
    )
    chat_inputs = [msg_input, chatbot, name_input, car_input, mileage_input, pref_input]
    msg_input.submit(chat_with_ai, inputs=chat_inputs, outputs=[msg_input, chatbot])
    submit_btn.click(chat_with_ai, inputs=chat_inputs, outputs=[msg_input, chatbot])

demo.launch(share=True)

In [ ]:
import json
import gradio as gr
import plotly.graph_objects as go
from google import genai
from google.genai import types
from google.colab import userdata

MODEL = "gemini-3.8-flash"   # 換成你帳號可用的 Gemini 模型名稱

# 1. 初始化
try:
    client = genai.Client(api_key=userdata.get('api'))
except Exception as e:
    print(f"API Key 讀取失敗: {e}")

# 2. 確認建立檔案
def confirm_profile(name, car, mileage, pref):
    if not name or not car:
        return "⚠️ 請至少填寫您的稱呼與車款！", gr.update(interactive=False)
    return (f"✅ 檔案建立成功！您好，{name}。\n系統已紀錄您的愛車為 {car}，請在右側開始對話。",
            gr.update(interactive=True, placeholder="請輸入您的問題並按 Enter 送出..."))

# 3. 工具函式
def _text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(c.get("text", "") if isinstance(c, dict) else (c if isinstance(c, str) else "") for c in content)
    if isinstance(content, dict):
        return content.get("text", "")
    return ""   # 圖表等元件不放進文字紀錄

SYSTEM_RULES = """
你是 TOYOTA 專屬的智能隨行顧問，語氣親切專業，使用繁體中文。
請「只」輸出一個 JSON 物件，格式如下，不要有其他文字：
{
  "answer": "給車主的文字回答（可用 Markdown 條列）",
  "chart": null 或
    {
      "type": "bar" | "line" | "pie",
      "title": "圖表標題",
      "x": ["類別或時間1", "類別或時間2", ...],
      "series": [{"name": "系列名稱", "values": [數字, 數字, ...]}],
      "y_label": "單位，例如 元／年、km、度"
    }
}
規則：
1. 只有在數字比較、費用試算、保養時程、長期成本、用電/油耗比較等「圖表有助於理解」時才提供 chart，否則 chart 設為 null。
2. 圖表數字一律是依車主檔案與一般行情做的「估算」，answer 中須說明假設（如年里程、油電價），並提醒實際以原廠報價為準。
3. 不可編造原廠優惠、保固條款或確切價格；不確定就說明並給出範圍。
4. pie 只用第一個 series；每個 series 的 values 長度必須等於 x 的長度。
"""

def make_figure(spec):
    try:
        x, series = spec["x"], spec["series"]
        t = spec.get("type", "bar")
        fig = go.Figure()
        if t == "pie":
            fig.add_trace(go.Pie(labels=x, values=series[0]["values"], hole=0.35))
        else:
            for s in series:
                if t == "line":
                    fig.add_trace(go.Scatter(x=x, y=s["values"], mode="lines+markers", name=s.get("name", "")))
                else:
                    fig.add_trace(go.Bar(x=x, y=s["values"], name=s.get("name", "")))
            fig.update_layout(yaxis_title=spec.get("y_label", ""), barmode="group")
        fig.update_layout(title=spec.get("title", ""), template="plotly_white", height=380,
                          margin=dict(l=40, r=20, t=60, b=40))
        return fig
    except Exception:
        return None

def parse_reply(raw):
    raw = (raw or "").strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        raw = raw[4:] if raw.lower().startswith("json") else raw
    try:
        data = json.loads(raw)
        return str(data.get("answer", "")), data.get("chart")
    except Exception:
        return raw, None

# 4. 對話邏輯
def chat_with_ai(user_message, history, name, car, mileage, pref):
    history = history or []
    if not user_message or not user_message.strip():
        return "", history

    prompt = f"""
車主檔案：
- 車主姓名：{name}
- 車款：{car}
- 目前里程數：{mileage}
- 用車習慣/喜好：{pref}

--- 歷史對話紀錄 ---
"""
    for m in history:
        t = _text(m.get("content"))
        if t:
            prompt += f"{'車主' if m.get('role') == 'user' else '顧問'}：{t}\n"
    prompt += f"車主：{user_message}\n請輸出 JSON："

    history = history + [{"role": "user", "content": user_message}]
    try:
        resp = client.models.generate_content(
            model=MODEL,
            contents=prompt,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_RULES,
                response_mime_type="application/json",
            ),
        )
        answer, chart = parse_reply(resp.text)
        history.append({"role": "assistant", "content": answer or "（沒有取得回覆）"})
        if chart:
            fig = make_figure(chart)
            if fig is not None:
                history.append({"role": "assistant", "content": gr.Plot(fig)})
    except Exception as e:
        history.append({"role": "assistant", "content": f"伺服器錯誤，請稍後再試。({e})"})
    return "", history

# 5. 介面
with gr.Blocks(theme=gr.themes.Base()) as demo:
    gr.Markdown("# 🚗 TOYOTA SmartLife 專屬智能顧問")
    gr.Markdown("請先在左側建立您的專屬檔案，接著即可在右側開始體驗客製化的 AI 隨行服務。")

    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 📋 步驟 1：建立車主檔案")
            name_input = gr.Textbox(label="您的稱呼", placeholder="例如：魏先生")
            car_input = gr.Textbox(label="您的愛車車款", placeholder="例如：bZ4X")
            mileage_input = gr.Textbox(label="目前大約里程數", placeholder="例如：23000")
            pref_input = gr.Textbox(label="您的用車習慣與喜好", placeholder="例如：每天通勤")
            confirm_btn = gr.Button("✅ 確認並建立檔案", variant="primary")
            status_text = gr.Textbox(label="系統狀態", interactive=False)

        with gr.Column(scale=2):
            gr.Markdown("### 💬 步驟 2：智能對話區")
            try:
                chatbot = gr.Chatbot(label="顧問回覆", height=550, type="messages")
            except TypeError:
                chatbot = gr.Chatbot(label="顧問回覆", height=550)

            gr.Examples(
                examples=["幫我試算接下來 5 年的用車成本，並畫圖比較",
                          "未來一年的保養時程與預估費用，用圖表呈現",
                          "油電車和純電車的年度能源成本比較"],
                inputs=[], label="試試看這些問法（複製貼上到輸入框）"
            ) if False else None

            with gr.Row():
                msg_input = gr.Textbox(
                    label="請輸入您的問題",
                    placeholder="請先在左側點擊「確認並建立檔案」...",
                    scale=4, interactive=False
                )
                submit_btn = gr.Button("📤 送出", scale=1)

    confirm_btn.click(confirm_profile,
                      inputs=[name_input, car_input, mileage_input, pref_input],
                      outputs=[status_text, msg_input])
    chat_inputs = [msg_input, chatbot, name_input, car_input, mileage_input, pref_input]
    msg_input.submit(chat_with_ai, inputs=chat_inputs, outputs=[msg_input, chatbot])
    submit_btn.click(chat_with_ai, inputs=chat_inputs, outputs=[msg_input, chatbot])

demo.launch(share=True)